## Task 1a Part 1： Crude DP with Approximation, implementation with loop and Precomputation
### DP setup
1) State definitition: **(m,r)** at time **i**, where **m** is the current liquidity memory before submission, **r** is the remaining shares to be subimitted $$ r_i=N-\sum_{j=0}^{j=i-1} n_j $$. 
2) The action and feasible action set: action at time **i** is the size of order submission **n_i**, feasible action set would be $$n_i<=r_i,\qquad n_i\in\mathbb Z_{\ge 0} $$
3) One step state transition function: from **i** to **i+1**, transition function for M is:
   $$ m_{i+1}=\left\lceil \frac{M_i+n_i}{2}\right\rceil $$
   transition funtion for **R** is
   $$ r_{i+1}=r_i-n_i $$
4) One-period reward:
   $$ reward_i=s_i=\left\lceil\bigl[1-\alpha M_i^\pi\bigr]n_i\right\rceil$$
5) Last period condition：at time **T**, all remaining orders should be submitted
$$ n_{T-1}=N-\sum_{i=0}^{T-2}n_i$$
equivalent to $$ r_{T-1}=n_{T-1}$$
6) Bellman recursion: Define **d(i,m,r)** as the maximum value-to-go at time interval **i**
$$ d(i,m,r)=\max_{n_i=0,\ldots,r_i}\left[s_i+d(i+1,m_{i+1},r_i-n_i)\right]$$

### Approximation and implementation in this part
1) Discretization

The algorithm define **n_m**,**n_r**,**n_n** as the number of equal space on **N** from 0 to 1000000.

Since **M** and **R** all have same order of magnitude as **N**，the initial values for the number of grids are **n_m**=100, **n_r**=100, and for discretization of actions (determining minimum unit of order sumbission), **n_n** is set to equal to **n_r** for simplicity.

The grid is not fine enoughwith stepsize around 1000, since implementation with loop is very time consuming, we fix the resolution issue with linear interpolation and finer grid in the **Part2** with vectorization.

The functiont **grid** produces **M_grid**, **R_grid** and **N_grid**

2) Bilinear Interpolation of value-to-go

When computing the value-to-go at time interval **i+1**, we need to update values of **M** and **R**, however, they are off the grid most of the time, so the stored value-to-go on the grid from previous interval in backward recursion appears insufficient.(In the simplified version however, with **n_n**=**n_r**, **R** value is always on the grid, so only interpolation on **M** is needed, defining **RegularGridInterpolator** is just for potential generalization purpose)

Suppose $$m_{lower}<=new\_m<=m_{upper}$$ $$r_{lower}<=new\_r<=r_{upper}$$ where the lower and upper values are both on the grid
Define the weights
$$ w_m = \frac{new\_m - m_{lower}}{m_{upper} - m_{lower}}, \quad w_r = \frac{new\_r - r_{lower}}{r_{upper} - r_{lower}} $$
therefore, the values of the four corners are
$$
\begin{aligned}
V_{LL} &= d(i+1, m_{lower}, r_{lower}) \\
V_{HL} &= d(i+1, m_{upper}, r_{lower}) \\
V_{LH} &= d(i+1, m_{lower}, r_{upper}) \\
V_{HH} &= d(i+1, m_{upper}, r_{upper})
\end{aligned}
$$
Therefore, we interpolate for the value across **M** and **R** under the function
$$
d_{interp}(i+1, new\_m, new\_r) = (1-w_m)(1-w_r)V_{LL} + w_m(1-w_r)V_{HL} + (1-w_m)w_r V_{LH} + w_m w_r V_{HH}
$$
Interpolation creates non-integer values for value-to-go function, in simulation, the filled sizes are rounded to integer

3) Precomputation

We precompute the values of **new_m** **new_r** and **reward(m,n)** and stored in the vector.

So that later given the current **m_i** and **r_i**'s indices in the grid, we looked up the matrix for the corresponding new values after different action **n_i**, and the reward value.

4) Implementation

The backward DP is naively implemented through loop, finding the value-to-go matrix, and then another function extracts the optimal schedule with a forward DP, to recover n for off-grid-values of **(m,r)**, bilinear interpolation is till necessary, candidates of **n** is taken from the N_grid.

Given the schedule, the execution is simulated with the exact formula without grid.

In [2]:
import csv
import sys
import numpy as np
import math
import matplotlib.pyplot as plt
import pandas as pd
import time

In [14]:
from scipy.interpolate import RegularGridInterpolator
# Below is the baseline implementation with loop and without precomputation on Task 1a
# Define the DP function
# Define the discretization function
def grid(N,n_m,n_r,n_n): # n_m as the number of spaces for liquidity Memory, n_r as the number of spaces for remaining size,n_n as number of spaces for order submission:
    M_grid=np.arange(0,N+N/n_m,N/n_m)
    R_grid=np.arange(0,N+N/n_r,N/n_r)
    N_grid=np.arange(0,N+N/n_n,N/n_n)
    return M_grid,R_grid,N_grid
# Define the transition function of M（original without precomputation)
def transition(m,n):
    m=math.ceil((m+n)/2)
    return m
# Define the reward function (original without precomputation)
def reward(n,new_m,alpha,pi):
    return math.ceil(max(0.0,1.0-alpha*new_m**pi)*n)
    
def DP_loop(N,T,pi,alpha,d:np.array):
    M_grid,R_grid,N_grid=grid(N,n_m,n_r,n_n)
    # Precomputation of the value of new_m, reward, and new_r
    new_m_mtrx=np.ceil((M_grid[:,None]+N_grid[None,:])/2) # n_m*n_n
    reward_mtrx=np.ceil(np.maximum(0,1-alpha*new_m_mtrx**pi)*N_grid[None,:]) # n_m*n_n
    new_r_mtrx=R_grid[:,None]-N_grid[None,:] #n_r*n_n, the new remaining shares value when n shares submitted, for all values on N_grid
    # Define the terminal Condition (submitting all remaining orders at T-1)
    for r_idx,r in enumerate(R_grid) :
        n=r
        for m_idx,m in enumerate(M_grid):
            new_m=new_m_mtrx[m_idx,r_idx] #Here since n=r and N and R's grid n_n=n_r, n_idx=r_idx
            d[T-1,m_idx,r_idx]=reward_mtrx[m_idx,r_idx]
    print(f't={T} completed')
    # Backward recursion
    for i in range(T-2,-1,-1):
        #Define interpolator for the period i+1
        interp = RegularGridInterpolator((M_grid, R_grid), d[i+1,:,:],bounds_error=False, fill_value=-np.inf)
        # loop over states (m,r)
        for r_idx,r in enumerate(R_grid):
            for m_idx,m in enumerate(M_grid):
                if r==0:
                    d[i,m_idx,r_idx]=0
                else:
                    best=-np.inf
                    for n_idx,n in enumerate(N_grid):
                        if n>r:
                            break
                        # Compute the new liquidity memory value after order submisssion
                        new_m=new_m_mtrx[m_idx,n_idx]
                        new_r=new_r_mtrx[r_idx,n_idx]
                        # finding the next period value-to-go 
                        if new_r==0: # interpolating only across grid of M
                            filled_later=np.interp(new_m,M_grid,d[i+1,:,0])
                        else:
                            # i+1 period's value-to-go is of the grid, interpolate
                            filled_later=interp([[new_m,new_r]])[0]
                        filled_now=reward_mtrx[m_idx,n_idx]
                        value=filled_now+filled_later
                        if value>best:
                            best=value
                    d[i,m_idx,r_idx]=best
        print(f't={i+1} completed')
    return d,M_grid,R_grid,N_grid

# Extract optimal schedule
def extract_schedule(d,N,T,alpha,pi,M_grid,R_grid,N_grid):
    schedule=[]
    m=0
    r=N
    for i in range(T):
        # Optimal action at T is to submit every shares left
        if i==T-1:
            n=r
            best_n=n
        else:
            interp = RegularGridInterpolator((M_grid, R_grid), d[i+1,:,:],bounds_error=False, fill_value=-np.inf)
            # if already all are submitted, n_i=0
            if r==0:
                n=0
                best_n=0
            else:
                best=-np.inf
                best_n=0
                for n_idx,n in enumerate(N_grid):
                    if n>r:
                        break
                    else:
                        new_m=transition(m,n)
                        new_r=r-n
                        if new_r==0:
                            filled_later=np.interp(new_m,M_grid,d[i+1,:,0])
                        else:
                            filled_later=interp([[new_m,new_r]])[0]
                        value=reward(n,new_m,alpha,pi)+filled_later
                        if value>best:
                            best=value
                            best_n=n
        schedule.append(best_n)
        m=transition(m,best_n)
        r=r-best_n
    return np.array(schedule,dtype=int)
def simulate_schedule(schedule, alpha, pi):
    """Simulate the liquidity-memory model for a given integer order schedule."""
    M = 0
    fills = []
    memory = []

    for n_i in schedule:
        n_i = int(n_i)
        M = math.ceil(0.5 * (M + n_i))
        traded = math.ceil(max(0.0, 1.0 - alpha * M**pi) * n_i)

        memory.append(M)
        fills.append(traded)

    return np.asarray(fills, dtype=int), np.asarray(memory, dtype=int)

In [15]:
# Implementation
t0=time.perf_counter()
N=100000
T=10
alpha=0.001
pi=0.5
n_m=100
n_r=100
n_n=100
d=np.full((T,n_m+1,n_r+1),-np.inf)
d,M_grid,R_grid,N_grid=DP_loop(N,T,pi,alpha,d)
schedule_app_loop=extract_schedule(d,N,T,alpha,pi,M_grid,R_grid,N_grid)
fills_app, memory_app = simulate_schedule(schedule_app_loop, alpha, pi)

for i in range(T):
    print(f"period {i}: order = {schedule_app_loop[i]:5d}, M = {memory_app[i]}, traded = {fills_app[i]}")
print("total traded =", fills_app.sum())
print(f'total runtime:{(time.perf_counter()-t0)}')

t=10 completed
t=9 completed
t=8 completed
t=7 completed
t=6 completed
t=5 completed
t=4 completed
t=3 completed
t=2 completed
t=1 completed
period 0: order = 12000, M = 6000, traded = 11071
period 1: order = 10000, M = 8000, traded = 9106
period 2: order =  9000, M = 8500, traded = 8171
period 3: order =  9000, M = 8750, traded = 8159
period 4: order =  9000, M = 8875, traded = 8153
period 5: order =  9000, M = 8938, traded = 8150
period 6: order =  9000, M = 8969, traded = 8148
period 7: order =  9000, M = 8985, traded = 8147
period 8: order = 10000, M = 9493, traded = 9026
period 9: order = 14000, M = 11747, traded = 12483
total traded = 90614
total runtime:495.818411400076


### Results stated above